# Theorem 2 — constructive existence

**Formal source:** [`../02_constructive_existence.md`](../02_constructive_existence.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(2)
count = 80000
source_a = rng.normal(-1, 0.5, count)
source_b = rng.normal(2, 0.75, count)
canonical_a = (source_a + 1) / 0.5
canonical_b = (source_b - 2) / 0.75
assert abs(canonical_a.mean()) < 0.02 and abs(canonical_b.mean()) < 0.02
assert abs(canonical_a.var() - 1) < 0.03 and abs(canonical_b.var() - 1) < 0.03
private = rng.normal(size=count)
missing = canonical_a + 0.5 * private + rng.normal(0, 0.2, count)
residual = missing - canonical_a - 0.5 * private
assert abs(residual.std() - 0.2) < 0.005
representation = {"shared": canonical_a, "private": private.copy(), "missing": missing}
assert "global_null" not in representation and np.array_equal(representation["private"], private)
print({"canonical_mean_gap": float(abs(canonical_a.mean() - canonical_b.mean())), "missing_residual_std": float(residual.std())})

In [ ]:
print('THEORY_DEMO_PASS::02_constructive_existence')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')